## 说明

### 项目概述
这个文件演示了**如何让AI代理在长时间对话中记住重要信息**，解决LLM（大型语言模型）的"健忘症"问题。就像人开会时间长了会忘记前面讨论的内容，AI也有类似的限制 - 它只能记住有限的对话历史（称为"token限制"）。

### 核心问题
- AI模型有输入长度限制（比如只能处理4096个词）
- 长时间对话会超出这个限制
- 简单截断对话会丢失重要上下文

### 解决方案
1. **聊天记录缩减**：自动将旧对话总结成简短摘要
2. **代理草稿板**：像便签本一样记录关键信息（用户偏好、已完成任务）

### 实现原理（简单版）
想象你和朋友讨论旅行计划：

- 你们聊了2小时，内容太多记不住
- 你决定：
    1. 把前面讨论的要点写在便签上（**代理草稿板**）
    2. 把详细讨论内容总结成几句话（**聊天记录缩减**）
- 这样即使忘记了细节，也能通过便签和摘要继续对话

# 使用语义内核中的代理草稿板进行聊天记录缩减

本笔记本展示了如何使用语义内核的聊天记录缩减功能以及代理草稿板来维护对话上下文。这对于构建能够处理长时间对话且不超出令牌限制的高效 AI 代理至关重要。

## 你将学到：
1. **聊天记录缩减**：如何自动总结对话记录以管理令牌使用
2. **代理草稿板**：一种用于跟踪用户偏好和已完成任务的持久化记忆系统
3. **令牌使用跟踪**：监控在启用和未启用记录缩减时令牌使用的变化

## 前置条件：
- 配置了环境变量的 Azure OpenAI 设置
- 理解之前课程中介绍的基本代理概念


## 导入所需的包


In [1]:
# Import necessary packages
import json
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display, HTML, Markdown
from typing import Annotated, Optional

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.completion_usage import CompletionUsage
from semantic_kernel.contents import FunctionCallContent, ChatHistorySummarizationReducer
from semantic_kernel.functions import kernel_function

## 理解代理记事本

### 什么是代理记事本？

**代理记事本**是一种持久化的记忆系统，代理可以用来：
- **跟踪已完成的任务**：记录为用户完成的事项
- **存储用户偏好**：记住用户的喜好、不喜欢的内容和需求
- **保持上下文**：在对话中保留重要信息以供随时访问
- **减少重复**：避免重复询问相同的问题

### 工作原理：
1. **写入操作**：代理在获取新信息后更新记事本
2. **读取操作**：代理在做决策时查阅记事本
3. **持久性**：即使聊天记录被缩减，信息仍然保留

可以将其视为代理的个人笔记本，用来补充对话历史记录。


## 环境配置


In [2]:
# Load environment variables
load_dotenv()

from openai import AsyncOpenAI
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
# 创建OpenAI聊天服务（使用GitHub的Inference API）
model_name = "gpt-4.1-mini"
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)
chat_service = OpenAIChatCompletion(
    # 指定要使用的模型ID（这里是通义千问的qwen-max）
    ai_model_id=model_name,
    # 传入之前创建的AsyncOpenAI客户端
    async_client=client,
)
print("✅ OpenAI service configured")

✅ OpenAI service configured


## 创建代理便签插件

此插件允许代理读取和写入一个持久化的便签文件。


In [3]:
# ======================
# 3. 代理草稿板插件
# ======================
class ScratchpadPlugin:
    """Plugin for managing agent scratchpad - a persistent memory for user preferences and completed tasks
    代理的"便签本" - 用于持久存储用户偏好和已完成任务
    - ScratchpadPlugin就像一个"智能便签本"
    - read_scratchpad()：查看便签本内容
    - update_scratchpad()：向便签本添加新信息
    - 便签本保存为Markdown文件，结构清晰易读
    - 为什么需要便签本？
    * AI模型有记忆限制，对话太长会被"遗忘"
    * 便签本帮助记住重要信息（如用户说"我喜欢海滩"）
    * 即使聊天历史被缩减，便签本内容依然保留
    """
    
    def __init__(self, filepath: str = "agent_scratchpad.md"):
        self.filepath = Path(filepath)   # 草稿板文件保存路
        # Initialize scratchpad if it doesn't exist
        # 如果文件不存在，创建新文件并初始化内容
        # 记录用户的喜好，如"喜欢海滩"、"预算3000美元"等
        # 记录已完成的工作，如"已创建巴厘岛行程
        if not self.filepath.exists():
            self.filepath.write_text("# 代理便签本\n\n## 用户偏好\n\n## 任务完成\n\n")
    
    # 读取当前便签本内容，获取用户旅行偏好和已完成任务
    @kernel_function(
        description="读取当前代理便签本以获取用户的旅行偏好和已完成任务"
    )
    def read_scratchpad(self) -> Annotated[str, "代理便签本的内容"]:
        """读取当前便签本的内容"""
        return self.filepath.read_text()
    
    # 更新便签本，添加新的用户旅行偏好或已完成任
    @kernel_function(
        description="用新用户的旅行偏好或已完成的任务更新代理记事本"
    )
    def update_scratchpad(
        self,
        category: Annotated[str, "要更新的类别：'偏好' 或 '任务'"],
        content: Annotated[str, "要添加的新内容"]
    ) -> Annotated[str, "更新确认"]:
        """Update the scratchpad with new information
        更新便签本"""
        # 读取当前内容
        current_content = self.filepath.read_text()
        # 获取当前时间（用于记录更新时间）
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
         # 根据类别更新相应部分
        if category.lower() == "偏好":
            # 在"用户偏好"部分添加新内容
            lines = current_content.split("\n")
            for i, line in enumerate(lines):
                if "## 用户偏好" in line:
                    # 在标题后插入新内容
                    lines.insert(i + 1, f"\n- [{timestamp}] {content}")
                    break
            current_content = "\n".join(lines)
        elif category.lower() == "任务":
            # 在"已完成任务"部分添加新内容
            # Find the tasks section and append
            lines = current_content.split("\n")
            for i, line in enumerate(lines):
                if "## 任务完成" in line:
                    lines.insert(i + 1, f"\n- [{timestamp}] {content}")
                    break
            current_content = "\n".join(lines)
        
        # 保存更新后的内容
        self.filepath.write_text(current_content)
        return f"✅ Scratchpad updated with {category}: {content}"

# Create the scratchpad plugin
# 创建便签本插件实例
scratchpad_plugin = ScratchpadPlugin("vacation_agent_scratchpad.md")
print("📝 创建了Scratchpad插件")

📝 创建了Scratchpad插件


## 初始化聊天记录缩减器

ChatHistorySummarizationReducer 会在对话历史超过阈值时自动生成摘要。


In [4]:
# ======================
# 4. 聊天记录缩减器
# ======================
# 配置缩减参数
REDUCER_TARGET_COUNT = 5  # 缩减后保留的对话消息数量
REDUCER_THRESHOLD = 15    # 触发缩减的对话消息数量阈值

# Create the history summarization reducer
# 创建聊天历史缩减器
# - ChatHistorySummarizationReducer就像"对话压缩器"
# - 工作原理：
#   1. 当对话超过15条消息(REDUCER_THRESHOLD)时
#   2. 自动将前10条消息总结成1-2条简洁摘要
#   3. 最终保留5条最新消息(REDUCER_TARGET_COUNT)
# - 好处：
#   * 减少AI模型的输入长度
#   * 节省token（相当于减少API调用费用）
#   * 保留对话关键信息
# - 就像把2小时会议录音总结成1页会议纪要
history_reducer = ChatHistorySummarizationReducer(
    service=chat_service,      # 使用的AI服务
    target_count=REDUCER_TARGET_COUNT,  # 目标保留消息数
    threshold_count=REDUCER_THRESHOLD,  # 触发缩减的阈值
)

print(f"🔄 Chat History Reducer configured:")
print(f"   - Reduction triggered at: {REDUCER_THRESHOLD} messages")
print(f"   - Reduces history to: {REDUCER_TARGET_COUNT} messages")

🔄 Chat History Reducer configured:
   - Reduction triggered at: 15 messages
   - Reduces history to: 5 messages


In [5]:
# ======================
# 5. Token跟踪器
# ======================
# Token tracking class
class TokenTracker:
    def __init__(self):
        self.history = []       # 存储每条消息的token使用
        self.total_usage = CompletionUsage()  # 累计token使用
        self.reduction_events = []  # 记录缩减发生的位置

    def add_usage(self, usage: CompletionUsage, message_num: int, thread_length: int = None):
        """添加token使用记录"""
        if usage:
            self.total_usage += usage
            # 创建详细的使用记录
            entry = {
                "message_num": message_num,
                "prompt_tokens": usage.prompt_tokens,  # 输入token
                "completion_tokens": usage.completion_tokens,  # 输出token
                "total_tokens": usage.prompt_tokens + usage.completion_tokens,
                "cumulative_tokens": self.total_usage.prompt_tokens + self.total_usage.completion_tokens,
                "thread_length": thread_length  # 当前对话历史长度
            }
            self.history.append(entry)

    def mark_reduction(self, message_num: int):
        """标记缩减事件发生的位置"""
        self.reduction_events.append(message_num)

    def display_chart(self):
        """显示一个图表，显示每个消息的令牌使用情况和减少的影响"""
        if not self.history:
            return

        # 创建HTML图表
        html = "<div style='font-family: monospace; background: #2d2d2d; color: #f0f0f0; padding: 15px; border-radius: 8px; border: 1px solid #444;'>"
        html += "<h4 style='color: #4fc3f7; margin-top: 0;'>📊 Token使用分析</h4>"
        html += "<pre style='color: #f0f0f0; margin: 0;'>"
        
        # Show prompt tokens per message to see reduction impact
        # 显示每条消息的输入token（显示对话上下文大小）
        html += "<span style='color: #81c784;'>每条消息的输入Token（显示对话上下文大小）:</span>\n"
        # 计算缩放比例，使图表适合显示
        max_prompt = max(h["prompt_tokens"] for h in self.history)
        scale = 50 / max_prompt if max_prompt > 0 else 1

        # 为每条消息创建可视化条形图
        for i, h in enumerate(self.history):
            bar_length = int(h["prompt_tokens"] * scale)
            bar = "█" * bar_length
            # 标记缩减事件
            reduction_marker = " <span style='color: #ff6b6b;'>← 缩减发生!</span>" if h["message_num"] in self.reduction_events else ""
            # 添加消息条目
            html += f"<span style='color: #aaa;'>消息 {h['message_num']:2d}:</span> <span style='color: #4fc3f7;'>{bar}</span> <span style='color: #ffd93d;'>{h['prompt_tokens']:,} tokens</span>{reduction_marker}\n"
        
        html += "\n</pre></div>"
        display(HTML(html))

        # Calculate reduction impact
        # 计算缩减效果
        if self.reduction_events:
            # Find the message before and after first reduction
            # 找出第一次缩减前后的token数量
            first_reduction_msg = self.reduction_events[0]
            before_reduction = None
            after_reduction = None

            for h in self.history:
                if h["message_num"] == first_reduction_msg - 1:
                    before_reduction = h["prompt_tokens"]
                elif h["message_num"] == first_reduction_msg:
                    after_reduction = h["prompt_tokens"]

            # 显示缩减效果
            if before_reduction and after_reduction:
                reduction_amount = before_reduction - after_reduction
                reduction_percent = (reduction_amount / before_reduction * 100)
                print(f"\n🔄 实际缩减效果:")
                print(f"缩减前输入tokens: {before_reduction:,}")
                print(f"缩减后输入tokens: {after_reduction:,}")
                print(f"节省tokens: {reduction_amount:,} ({reduction_percent:.1f}%)")

# Display function for clean output


def display_message(role: str, content: str, color: str = "#2E8B57"):
    """显示具有良好格式的消息，可以在浅色和深色主题中工作"""
    # Use a semi-transparent background that adapts to the theme
    html = f"""
    <div style='
        margin: 10px 0; 
        padding: 12px 15px; 
        border-left: 4px solid {color}; 
        background: rgba(128, 128, 128, 0.1); 
        border-radius: 4px;
        color: inherit;
    '>
        <strong style='color: {color}; font-size: 14px;'>{role}:</strong><br>
        <div style='margin-top: 8px; white-space: pre-wrap; color: inherit; font-size: 14px;'>{content}</div>
    </div>
    """
    display(HTML(html))


# Initialize token tracker
# 创建token跟踪器
token_tracker = TokenTracker()
print("📊 Token tracking initialized")

📊 Token tracking initialized


## 创建度假规划助手

此助手将通过便笺功能帮助用户规划度假，同时保持上下文连贯性。


In [6]:
# ======================
# 6. 创建度假规划代理
agent = ChatCompletionAgent(
    service=chat_service,
    name="VacationPlannerAgent",
    instructions="""
你是一个度假规划助手，帮助用户规划完美假期。

📌 便签本使用规则（必须遵守）：
1. 开始对话时：立即调用read_scratchpad()检查现有偏好
2. 获取新偏好时：立即调用update_scratchpad()更新'preferences'
3. 完成任务时：立即调用update_scratchpad()更新'tasks'
4. 创建新行程前：总是先调用read_scratchpad()

📌 何时更新便签本示例：
- 用户说"我喜欢海滩" → update_scratchpad('preferences', '喜欢海滩目的地')
- 用户说"预算是3000美元" → update_scratchpad('preferences', '预算: 每人3000美元/周')
- 创建行程后 → update_scratchpad('tasks', '已创建巴厘岛海滩度假行程')

📌 规划流程：
1. 先读取便签本
2. 如果没有偏好，询问用户
3. 更新便签本记录新信息
4. 创建详细行程
5. 更新便签本记录已完成任务

📌 重要：总是明确说明你在查看或更新便签本。
    """,
    plugins=[scratchpad_plugin],
)

print("🤖 Vacation Planning Agent created with enhanced scratchpad instructions")

🤖 Vacation Planning Agent created with enhanced scratchpad instructions


## 运行度假计划对话

现在让我们完整演示一次对话，包括以下内容：
1. 初始计划请求
2. 偏好收集
3. 行程创建
4. 更改地点
5. 聊天记录缩减
6. 使用便签


In [7]:
# ======================
# 8. 模拟度假规划对话
# ======================
user_inputs = [
"我想规划一次度假，你能帮我吗？",
"我喜欢有美丽食物和文化的海滩目的地。我喜欢水上运动、探索当地市场和尝试正宗美食。我的预算是每人3000美元一周。",
"听起来很完美！请为我创建一个详细的巴厘岛行程。",
"实际上，我改变主意了。我更喜欢去希腊群岛。你能创建一个新的行程吗？",
"那里的天气怎么样？",
"我应该带什么？",
"有什么需要注意的文化习俗吗？",
"最好的交通方式是什么？"
]


async def run_vacation_planning():
    """Run the vacation planning conversation with token tracking and history reduction
    运行度假规划对话，包含token跟踪和历史缩减"""

    # Create thread with history reducer
    # 创建带缩减功能的对话线程
    thread = ChatHistoryAgentThread(chat_history=history_reducer)
    message_count = 0
    scratchpad_operations = 0  # 跟踪便签本操作次数

    print("🚀 Starting Vacation Planning Session\n")

    # Process conversation
    # 处理每条用户输入
    for i, user_input in enumerate(user_inputs):
        message_count += 1
        display_message("User", user_input, "#4fc3f7")  # Blue for user

        # Get agent response
        # 获取代理响应
        full_response = ""
        usage = None
        function_calls = []  # 跟踪函数调用

        # 调用代理获取响应（支持流式响应）
        async for response in agent.invoke(
            messages=user_input,
            thread=thread,
        ):
            if response.content:
                full_response += str(response.content)
            if response.metadata.get("usage"):
                usage = response.metadata["usage"]
            thread = response.thread

        # 显示代理响应
        display_message(f"{agent.name}", full_response,
                        "#81c784")  # Green for agent

        # Track tokens with thread length
        # 跟踪token使用情况
        if usage:
            token_tracker.add_usage(usage, message_count, len(thread))

        # Check thread status and look for scratchpad operations
        print(f"📝 线程有{len(thread)}条消息")

        # Count scratchpad operations in this turn
        # 检查并显示便签本操作
        turn_scratchpad_ops = 0
        async for msg in thread.get_messages():
            if hasattr(msg, 'items') and msg.items:
                content_items = list(msg.items) # 将响应项转换为列表进行遍历
                for item in content_items:
                    if isinstance(item, FunctionCallContent):
                        function_name = item.function_name
                        if function_name == 'read_scratchpad' or function_name == 'update_scratchpad':
                            turn_scratchpad_ops += 1
            # if hasattr(msg, 'content') and msg.content:
            #     content_str = str(msg.content)
            #     if 'read_scratchpad' in content_str or 'update_scratchpad' in content_str:
            #         turn_scratchpad_ops += 1

        if turn_scratchpad_ops > scratchpad_operations:
            print(f"   📝 检测到便签本操作: 新增 {turn_scratchpad_ops - scratchpad_operations} 次操作")
            scratchpad_operations = turn_scratchpad_ops

        # Show message types for first message
        # 显示第一条消息的类型（仅第一次）
        if i == 0:
            message_types = []
            async for msg in thread.get_messages():
                msg_type = msg.role.value if hasattr(
                    msg.role, 'value') else str(msg.role)
                message_types.append(msg_type)
            print(f"   消息类型: {message_types[:10]}..." if len(message_types) > 10 else f"   消息类型: {message_types}")
        
        # Check if reduction should happen
        # 检查是否需要缩减
        if len(thread) > REDUCER_THRESHOLD:
            print(f"   ⚠️ 对话历史 ({len(thread)}) 超过阈值 ({REDUCER_THRESHOLD})")
            
            # Attempt reduction
            is_reduced = await thread.reduce()
            if is_reduced:
                print(f"\n🔄 历史已缩减! 对话历史现在有 {len(thread)} 条消息\n")
                token_tracker.mark_reduction(message_count + 1)

                # Show summary if available
                async for msg in thread.get_messages():
                    if msg.metadata and msg.metadata.get("__summary__"):
                        display_message("System Summary", str(
                            msg.content), "#ff6b6b")
                        break

    # 显示token使用图表
    print("\n--- Token使用分析 ---")
    token_tracker.display_chart()
    
    # 显示最终便签本内容
    print("\n--- 最终便签本内容 ---")
    scratchpad_contents = scratchpad_plugin.read_scratchpad()
    display(Markdown(scratchpad_contents))
    
    print(f"\n📊 总便签本操作次数: {scratchpad_operations}")
    
    return thread

# Run the conversation
thread = await run_vacation_planning()

🚀 Starting Vacation Planning Session



📝 线程有4条消息
   📝 检测到便签本操作: 新增 1 次操作
   消息类型: ['user', 'assistant', 'tool', 'assistant']


📝 线程有8条消息
   📝 检测到便签本操作: 新增 1 次操作


📝 线程有14条消息
   📝 检测到便签本操作: 新增 2 次操作


📝 线程有21条消息
   📝 检测到便签本操作: 新增 3 次操作
   ⚠️ 对话历史 (21) 超过阈值 (15)

🔄 历史已缩减! 对话历史现在有 8 条消息



...

📝 线程有12条消息


📝 线程有14条消息


📝 线程有16条消息
   ⚠️ 对话历史 (16) 超过阈值 (15)

--- Token使用分析 ---



🔄 实际缩减效果:
缩减前输入tokens: 1,476
缩减后输入tokens: 1,102
节省tokens: 374 (25.3%)

--- 最终便签本内容 ---



📊 总便签本操作次数: 7


## 分析结果

让我们来分析一下在对话中发生了什么：


In [10]:
# Analyze token usage
print("📊 Total Token Usage Summary\n")
print(f"Total Prompt Tokens: {token_tracker.total_usage.prompt_tokens:,}")
print(
    f"Total Completion Tokens: {token_tracker.total_usage.completion_tokens:,}")
print(
    f"Total Tokens Used: {token_tracker.total_usage.prompt_tokens + token_tracker.total_usage.completion_tokens:,}")

print("\n💡 Note: The reduction impact is shown in the chart above.")
print("Look for the dramatic drop in prompt tokens after the REDUCTION marker.")
print("This shows how chat history summarization reduces the context size for future messages.")

📊 Total Token Usage Summary

Total Prompt Tokens: 17,563
Total Completion Tokens: 4,407
Total Tokens Used: 21,970

💡 Note: The reduction impact is shown in the chart above.
Look for the dramatic drop in prompt tokens after the REDUCTION marker.
This shows how chat history summarization reduces the context size for future messages.


## 关键要点

### 1. 聊天记录缩减
- **自动触发**：当消息数量超过阈值时会进行缩减
- **节省 Token**：摘要后显著减少 Token 使用量
- **保留上下文**：摘要中保留重要信息

### 2. 代理记事本的优势
- **持久记忆**：用户偏好在记录缩减后仍然保留
- **任务跟踪**：代理能够记录已完成的工作
- **提升体验**：无需重复说明偏好

### 3. Token 使用模式
- **线性增长**：每条消息都会增加 Token 数量
- **显著下降**：缩减后 Token 数量大幅减少
- **可持续对话**：在限制范围内实现更长时间的互动


## 清理

清理在本次演示过程中创建的临时文件：


In [ ]:
# Optional: Clean up the scratchpad file
# Uncomment the next line to delete the scratchpad
# Path("vacation_agent_scratchpad.md").unlink(missing_ok=True)

print("✅ Demo complete! The scratchpad file 'vacation_agent_scratchpad.md' has been preserved for your review.")

# 概要

恭喜你！你已经成功实现了一个具备高级上下文管理能力的AI代理：

## 你学到了什么：
- **聊天记录缩减**：自动总结对话内容以管理令牌限制
- **代理记事本**：实现用户偏好和已完成任务的持久记忆
- **令牌管理**：在长对话中跟踪并优化令牌使用
- **上下文保留**：在对话缩减过程中保持重要信息

## 实际应用场景：
- **客服机器人**：在多个会话中记住客户偏好
- **个人助理**：跟踪正在进行的项目和用户习惯
- **教育导师**：记录学生的学习进度和偏好
- **医疗助手**：在尊重令牌限制的同时保留患者历史记录

## 下一步：
- 实现更复杂的记事本结构
- 为多用户场景添加数据库存储
- 创建针对特定领域的自定义缩减策略
- 与向量数据库结合进行语义记忆搜索
- 构建能够在数天后恢复对话并保持完整上下文的代理



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保准确性，但请注意，自动翻译可能包含错误或不准确之处。应以原始语言的文档作为权威来源。对于关键信息，建议使用专业人工翻译。对于因使用本翻译而引起的任何误解或误读，我们概不负责。
